In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

In [2]:
import tensorflow as tf
from typing import Any

2026-01-27 16:31:24.704190: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-27 16:31:24.725946: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-27 16:31:24.732400: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-27 16:31:24.749287: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-27 16:31:26.418668: W tensorflow/compiler/tf2

In [3]:
from mobilenetv2ssd.models.ssd.ops.postprocess_tf import *

In [4]:
from mobilenetv2ssd.core.config import load_config

In [5]:
main_cfg_path = "configs/train/default.yaml"
eval_path = "configs/eval/default.yaml"

In [6]:
config = load_config(main_cfg_path,eval_path)

In [7]:
config['eval']

{'dataset_split': 'val',
 'input': [300, 300],
 'decode': {'variances': [0.1, 0.2],
  'priors_format': 'cxcywh',
  'output_format': 'xyxy',
  'clip_boxes': True,
  'boxes_normalized': True,
  'use_sigmoid': False,
  'root': '/mnt/d/dev/MobileNetV2-SSD/datasets/VOCdevkit/labels/voc_labels.txt'},
 'nms': {'iou_threshold': 0.5,
  'score_threshold': 0.05,
  'max_detections_per_class': 50,
  'max_detections_per_image': 100,
  'per_class_top_k': 100},
 'metrics': {'voc_ap_50': {'type': 'voc_ap',
   'iou_thresholds': [0.5, 0.75],
   'use_07_metric': False},
  'coco_map': {'type': 'coco_map',
   'iou_thresholds': [0.5, 0.75],
   'use_07_metric': False}},
 'visualization': {'enabled': False,
  'max_images': 16,
  'output_dir': '/mnt/d/dev/MobileNetV2-SSD/eval_vis'}}

In [25]:
def _read_eval_config(config: dict[str, Any]):
    eval_opts = config['eval']
    nms_config = eval_opts['nms']
    decode_config = eval_opts['decode']
    eval_config = {
        
        'iou_threshold': nms_config.get('iou_threshold', 0.5),
        'score_threshold': nms_config.get('score_threshold', 0.05),
        'max_detections_per_class': nms_config.get('max_detections_per_class', 50),
        'max_detections_per_image': nms_config.get('max_detections_per_image', 100),
        'per_class_top_k': nms_config.get('per_class_top_k', 100),
        'class_file': decode_config.get('root', None),
        'variances': tf.constant(list(decode_config.get('variances', [0.1,0.2])), dtype = tf.float32),
        'use_sigmoid': decode_config.get('use_sigmoid', False),
        'input_size': {'image_height': eval_opts.get('input', 0)[0],'image_width': eval_opts.get('input', 0)[1]}
    }
    
    return eval_config

In [26]:
eval_config = _read_eval_config(config)

In [27]:
eval_config

{'iou_threshold': 0.5,
 'score_threshold': 0.05,
 'max_detections_per_class': 50,
 'max_detections_per_image': 100,
 'per_class_top_k': 100,
 'class_file': '/mnt/d/dev/MobileNetV2-SSD/datasets/VOCdevkit/labels/voc_labels.txt',
 'variances': <tf.Tensor: shape=(2,), dtype=float32, numpy=array([0.1, 0.2], dtype=float32)>,
 'use_sigmoid': False,
 'input_size': {'image_height': 300, 'image_width': 300}}

In [20]:
deploy_config = _read_deploy_config(config)

In [21]:
deploy_config

{'input_size': {'image_height': 300, 'image_width': 300},
 'iou_thresh': 0.5,
 'variances': <tf.Tensor: shape=(2,), dtype=float32, numpy=array([0.1, 0.2], dtype=float32)>,
 'max_detection': 3,
 'per_class_top_k': 100,
 'score_thresh': 0.3,
 'num_classes': 3,
 'labels_map': '/mnt/d/dev/MobileNetV2-SSD/datasets/VOCdevkit/labels/voc_labels.txt',
 'use_sigmoid': False}

In [14]:
from pathlib import Path
def _load_label_map(label_file_path: str, use_sigmoid: bool = False):
    
    label_file_path = Path(label_file_path)
    
    with open(label_file_path, "r") as f:
        labels = [line.strip() for line in f.readlines() if line.strip()]

    # Build lookup
    if use_sigmoid:
        label_dict = {idx: name for idx, name in enumerate(labels)}
    else:
        # Softmax so labels begin at 1
        label_dict = {idx + 1: name for idx, name in enumerate(labels)}
        label_dict[0] = "background"
        
    return label_dict

In [15]:
def _decode_class_names(class_id_tensor: tf.Tensor, labels: dict[int,str]):
    
    labels_list = tf.constant([labels[key] for key in sorted(labels)], dtype=tf.string)
    decoded_classes = tf.gather(labels_list,class_id_tensor)
    
    return decoded_classes

In [17]:
_load_label_map(eval_config['class_file'],use_sigmoid = eval_config['use_sigmoid'])

{1: 'aeroplane',
 2: 'bicycle',
 3: 'bird',
 4: 'boat',
 5: 'bottle',
 6: 'bus',
 7: 'car',
 8: 'cat',
 9: 'chair',
 10: 'cow',
 11: 'diningtable',
 12: 'dog',
 13: 'horse',
 14: 'motorbike',
 15: 'person',
 16: 'pottedplant',
 17: 'sheep',
 18: 'sofa',
 19: 'train',
 20: 'tvmonitor',
 0: 'background'}

In [28]:
def build_decoded_boxes(config: dict[str,Any], predicted_offsets: tf.Tensor, predicted_logits: tf.Tensor, priors: tf.Tensor):
    eval_config = _read_eval_config(config)

    # Decoding boxes
    nmsed_boxes,nmsed_scores, nmsed_classes, valid_detections = decode_and_nms(predicted_offsets = predicted_offsets, predicted_logits = predicted_logits, priors = priors, variances = eval_config['variances'],scores_thresh = eval_config['score_threshold'], iou_thresh = eval_config['iou_threshold'], top_k = eval_config['per_class_top_k'], max_detections = eval_config['max_detections_per_image'],image_meta = eval_config['input_size'],use_sigmoid = eval_config['use_sigmoid'])   

    # Getting the classes
    classes = _load_label_map(eval_config['class_file'],use_sigmoid = eval_config['use_sigmoid'])

    decoded_classes = _decode_class_names(nmsed_classes, classes)

    return nmsed_boxes, nmsed_scores, nmsed_classes, decoded_classes, classes

In [29]:
pred_loc = tf.constant(
    [
  [  # batch 0
    [ 0.0,  0.0,  0.0,  0.0],   # anchor 0
    [ 0.2,  0.0,  0.0,  0.0],   # anchor 1
    [ 0.0,  0.0,  0.5,  0.0],   # anchor 2
    [ 0.0, -0.2,  0.0, -0.5],   # anchor 3
  ]
]
)

pred_logits = tf.constant(
    [
  [  # batch 0
    [0.1,  2.0,  0.0],   # anchor 0
    [0.0,  0.5,  3.0],   # anchor 1
    [0.5,  1.5, -1.0],   # anchor 2
    [0.2, -0.5,  0.0],   # anchor 3
  ]
]
)

priors = tf.constant(
    [
  [0.25, 0.25, 0.2, 0.2],  # anchor 0 (top-left)
  [0.75, 0.25, 0.2, 0.2],  # anchor 1 (top-right)
  [0.25, 0.75, 0.2, 0.2],  # anchor 2 (bottom-left)
  [0.75, 0.75, 0.2, 0.2],  # anchor 3 (bottom-right)
]
)

In [30]:
build_decoded_boxes(config,pred_loc,pred_logits, priors)

(<tf.Tensor: shape=(1, 100, 4), dtype=float32, numpy=
 array([[[ 45.     , 196.2    , 105.     , 256.2    ],
         [ 45.     ,  45.     , 105.     , 105.     ],
         [195.     ,  41.84487, 255.     , 108.15513],
         [196.65488, 195.     , 250.94511, 255.     ],
         [196.65488, 195.     , 250.94511, 255.     ],
         [ 45.     ,  45.     , 105.     , 105.     ],
         [ 45.     , 196.2    , 105.     , 256.2    ],
         [195.     ,  41.84487, 255.     , 108.15513],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [  0.     ,   0.     ,   0.     ,   0.     ],
         [ 